## test of the concept

In [1]:
class Participant:
    """A student: keeps its numbers private, only shares a result."""
    def __init__(self, name, data):
        self.name = name
        self.data = data            # private: never leaves this object

    def train(self):
        result = sum(self.data) / len(self.data)
        print(f"{self.name} trained locally -> {result}")
        return result


class Coordinator:
    """The teacher: collects the results and averages them."""
    def __init__(self):
        self.participants = []

    def register(self, participant):
        self.participants.append(participant)
        print(f"Registered {participant.name}")

    def run(self):
        results = [p.train() for p in self.participants]
        global_model = sum(results) / len(results)
        print(f"Global model = {global_model}")
        return global_model


coordinator = Coordinator()
coordinator.register(Participant("A", [8, 10, 12]))
coordinator.register(Participant("B", [18, 20, 22]))
coordinator.register(Participant("C", [28, 30, 32]))
coordinator.run()

Registered A
Registered B
Registered C
A trained locally -> 10.0
B trained locally -> 20.0
C trained locally -> 30.0
Global model = 20.0


20.0

# PHASE 1

In [9]:
from dataclasses import dataclass
from abc import ABC, abstractmethod

## messages that will travel between the teacher and the students

In [8]:
# ---------- Box 8: the messages ----------

@dataclass
class ModelParameters:
    """The model itself: a list of numbers."""
    weights: list[float]


@dataclass
class NodeInfo:
    """A node's ID card."""
    node_id: str
    address: str = "local"   # not used until Phase 2


@dataclass
class RoundConfig:
    """The server's note to a node at the start of a round."""
    exp_id: str
    round_num: int
    learning_rate: float
    global_model: ModelParameters


@dataclass
class ModelUpdate:
    """A node's answer after training."""
    exp_id: str
    node_id: str
    round_num: int
    local_model: ModelParameters
    dataset_size: int


if __name__ == "__main__":
    # A quick check: create one note and one answer, and print them
    note = RoundConfig(exp_id="exp-1", round_num=1, learning_rate=0.5,
                       global_model=ModelParameters([0.0]))
    answer = ModelUpdate(exp_id="exp-1", node_id="A", round_num=1,
                         local_model=ModelParameters([5.0]), dataset_size=3)
    print(note)
    print(answer)

RoundConfig(exp_id='exp-1', round_num=1, learning_rate=0.5, global_model=ModelParameters(weights=[0.0]))
ModelUpdate(exp_id='exp-1', node_id='A', round_num=1, local_model=ModelParameters(weights=[5.0]), dataset_size=3)


## worker node

In [7]:
class WorkerNode:
    """A student: keeps its data private, returns only a model update."""

    def __init__(self, info: NodeInfo, private_data: list[float]):
        self.info = info
        self._private_data = private_data   # "_" means private, like the "-" in the diagram

    def train_local_model(self, config: RoundConfig) -> ModelUpdate:
        # 1. compute this node's own answer from its private data
        local_mean = sum(self._private_data) / len(self._private_data)

        # 2. move each weight of the global model toward that answer
        new_weights = [w + config.learning_rate * (local_mean - w)
                       for w in config.global_model.weights]

        print(f"  {self.info.node_id}: got {config.global_model.weights} "
              f"-> trained -> {[round(w, 2) for w in new_weights]}")

        # 3. send back only the result, never the data
        return ModelUpdate(exp_id=config.exp_id,
                           node_id=self.info.node_id,
                           round_num=config.round_num,
                           local_model=ModelParameters(new_weights),
                           dataset_size=len(self._private_data))


if __name__ == "__main__":
    # A quick check: one node, one note, one answer
    node_a = WorkerNode(NodeInfo("A"), private_data=[8, 10, 12])

    note = RoundConfig(exp_id="exp-1", round_num=1, learning_rate=0.5,
                       global_model=ModelParameters([0.0]))
    answer = node_a.train_local_model(note)
    print(answer)

  A: got [0.0] -> trained -> [5.0]
ModelUpdate(exp_id='exp-1', node_id='A', round_num=1, local_model=ModelParameters(weights=[5.0]), dataset_size=3)


## AggregationStrategy and FedAvgStrategy 

In [12]:
class AggregationStrategy(ABC):
    """The job description: 'combine the nodes' answers into one model'."""

    @abstractmethod
    def aggregate(self, updates: list[ModelUpdate]) -> ModelParameters:
        ...


class FedAvgStrategy(AggregationStrategy):
    """The one doing the job: a weighted average (more data = more weight)."""

    def aggregate(self, updates: list[ModelUpdate]) -> ModelParameters:
        total_samples = sum(u.dataset_size for u in updates)
        num_weights = len(updates[0].local_model.weights)

        new_weights = []
        for i in range(num_weights):
            weighted_sum = sum(u.local_model.weights[i] * u.dataset_size for u in updates)
            new_weights.append(weighted_sum / total_samples)

        return ModelParameters(new_weights)

## testing the calculator

In [13]:
nodes = [WorkerNode(NodeInfo("A"), [8, 10, 12]),
         WorkerNode(NodeInfo("B"), [18, 20, 22]),
         WorkerNode(NodeInfo("C"), [28, 30, 32])]

note = RoundConfig(exp_id="exp-1", round_num=1, learning_rate=1.0,
                   global_model=ModelParameters([0.0]))
answers = [node.train_local_model(note) for node in nodes]

strategy = FedAvgStrategy()
print("Global model =", strategy.aggregate(answers))

  A: got [0.0] -> trained -> [10.0]
  B: got [0.0] -> trained -> [20.0]
  C: got [0.0] -> trained -> [30.0]
Global model = ModelParameters(weights=[20.0])


## NodeRegistry

In [14]:
class NodeRegistry:
    """The class list: remembers every registered node."""

    def __init__(self):
        self._node_infos: dict[str, NodeInfo] = {}        # node_id -> ID card
        self._active_workers: dict[str, WorkerNode] = {}  # node_id -> the node itself

    def register(self, info: NodeInfo, worker: WorkerNode) -> bool:
        if info.node_id in self._node_infos:
            print(f"Node {info.node_id} is already registered")
            return False
        self._node_infos[info.node_id] = info
        self._active_workers[info.node_id] = worker
        print(f"Registered node {info.node_id}")
        return True

    def get_active_workers(self) -> list[WorkerNode]:
        return list(self._active_workers.values())

## test node registry

In [16]:
registry = NodeRegistry()

node_a = WorkerNode(NodeInfo("A"), [8, 10, 12])
node_b = WorkerNode(NodeInfo("B"), [18, 20, 22])
node_c = WorkerNode(NodeInfo("C"), [28, 30, 32])

registry.register(node_a.info, node_a)
registry.register(node_b.info, node_b)
registry.register(node_c.info, node_c)

# try to register A a second time
result = registry.register(node_a.info, node_a)
print("Second registration worked?", result)

# who is on the list?
print("Active nodes:", [w.info.node_id for w in registry.get_active_workers()])

Registered node A
Registered node B
Registered node C
Node A is already registered
Second registration worked? False
Active nodes: ['A', 'B', 'C']


## experament and exp status

In [17]:
from enum import Enum
from dataclasses import field


class ExperimentStatus(Enum):
    """The only four stages a job can be in."""
    CREATED = "CREATED"
    RUNNING = "RUNNING"
    COMPLETED = "COMPLETED"
    FAILED = "FAILED"


@dataclass
class Experiment:
    """The record of one training job."""
    exp_id: str
    total_rounds: int
    learning_rate: float
    global_model: ModelParameters
    status: ExperimentStatus = ExperimentStatus.CREATED
    current_round: int = 0
    round_updates: list[ModelUpdate] = field(default_factory=list)

In [18]:
exp = Experiment(exp_id="exp-1", total_rounds=3, learning_rate=0.5,
                 global_model=ModelParameters([0.0]))
print(exp)
print("Status:", exp.status.value)

# pretend the job has started
exp.status = ExperimentStatus.RUNNING
exp.current_round = 1
print("Status now:", exp.status.value, "| round", exp.current_round, "of", exp.total_rounds)

Experiment(exp_id='exp-1', total_rounds=3, learning_rate=0.5, global_model=ModelParameters(weights=[0.0]), status=<ExperimentStatus.CREATED: 'CREATED'>, current_round=0, round_updates=[])
Status: CREATED
Status now: RUNNING | round 1 of 3


## exp manager

In [19]:
class ExperimentManager:
    """Runs the training jobs, round after round."""

    def __init__(self, registry: NodeRegistry, strategy: AggregationStrategy):
        self._experiments: dict[str, Experiment] = {}
        self._registry = registry
        self._strategy = strategy
        self._next_id = 1

    def create_experiment(self, rounds: int, lr: float) -> str:
        exp_id = f"exp-{self._next_id}"
        self._next_id += 1
        self._experiments[exp_id] = Experiment(exp_id=exp_id, total_rounds=rounds,
                                               learning_rate=lr,
                                               global_model=ModelParameters([0.0]))
        print(f"Created {exp_id}: {rounds} rounds, learning rate {lr}")
        return exp_id

    def start_experiment(self, exp_id: str) -> Experiment:
        exp = self.get_status(exp_id)
        if exp.status != ExperimentStatus.CREATED:
            raise ValueError(f"{exp_id} is {exp.status.value}, only CREATED experiments can start")

        workers = self._registry.get_active_workers()
        if not workers:
            raise ValueError("no nodes are registered")

        exp.status = ExperimentStatus.RUNNING
        print(f"Started {exp_id} with {len(workers)} nodes")
        try:
            for r in range(1, exp.total_rounds + 1):
                exp.current_round = r
                print(f"Round {r}")

                # 1. write the note for this round
                config = RoundConfig(exp_id=exp_id, round_num=r,
                                     learning_rate=exp.learning_rate,
                                     global_model=exp.global_model)

                # 2. send it to every node and collect the answers
                exp.round_updates = [w.train_local_model(config) for w in workers]

                # 3. let the calculator combine the answers
                exp.global_model = self._strategy.aggregate(exp.round_updates)
                print(f"  global model = {[round(w, 2) for w in exp.global_model.weights]}")

            exp.status = ExperimentStatus.COMPLETED
        except Exception as e:
            exp.status = ExperimentStatus.FAILED
            print(f"{exp_id} failed: {e}")
        return exp

    def get_status(self, exp_id: str) -> Experiment:
        if exp_id not in self._experiments:
            raise ValueError(f"unknown experiment {exp_id}")
        return self._experiments[exp_id]

In [20]:
# set up: a class list with three nodes, and the FedAvg calculator
registry = NodeRegistry()
for node in [WorkerNode(NodeInfo("A"), [8, 10, 12]),
             WorkerNode(NodeInfo("B"), [18, 20, 22]),
             WorkerNode(NodeInfo("C"), [28, 30, 32])]:
    registry.register(node.info, node)

manager = ExperimentManager(registry, FedAvgStrategy())

# create a job, run it, check it
exp_id = manager.create_experiment(rounds=3, lr=0.5)
print("Status before:", manager.get_status(exp_id).status.value)

manager.start_experiment(exp_id)

exp = manager.get_status(exp_id)
print("Status after:", exp.status.value, "| final model:", exp.global_model.weights)

# a mistake: start the same job again
try:
    manager.start_experiment(exp_id)
except ValueError as e:
    print("Error:", e)

Registered node A
Registered node B
Registered node C
Created exp-1: 3 rounds, learning rate 0.5
Status before: CREATED
Started exp-1 with 3 nodes
Round 1
  A: got [0.0] -> trained -> [5.0]
  B: got [0.0] -> trained -> [10.0]
  C: got [0.0] -> trained -> [15.0]
  global model = [10.0]
Round 2
  A: got [10.0] -> trained -> [10.0]
  B: got [10.0] -> trained -> [15.0]
  C: got [10.0] -> trained -> [20.0]
  global model = [15.0]
Round 3
  A: got [15.0] -> trained -> [12.5]
  B: got [15.0] -> trained -> [17.5]
  C: got [15.0] -> trained -> [22.5]
  global model = [17.5]
Status after: COMPLETED | final model: [17.5]
Error: exp-1 is COMPLETED, only CREATED experiments can start


## FLServer

In [21]:
class FLServer:
    """The teacher: owns the two assistants and passes each request to the right one."""

    def __init__(self):
        self.registry = NodeRegistry()
        self.experiment_manager = ExperimentManager(self.registry, FedAvgStrategy())

    def register_node(self, info: NodeInfo, worker: WorkerNode) -> bool:
        return self.registry.register(info, worker)

    def create_experiment(self, rounds: int, lr: float) -> str:
        return self.experiment_manager.create_experiment(rounds, lr)

    def start_experiment(self, exp_id: str) -> Experiment:
        return self.experiment_manager.start_experiment(exp_id)

    def get_experiment_status(self, exp_id: str) -> Experiment:
        return self.experiment_manager.get_status(exp_id)

In [22]:
server = FLServer()

for node in [WorkerNode(NodeInfo("A"), [8, 10, 12]),
             WorkerNode(NodeInfo("B"), [18, 20, 22]),
             WorkerNode(NodeInfo("C"), [28, 30, 32])]:
    server.register_node(node.info, node)

exp_id = server.create_experiment(rounds=1, lr=1.0)
server.start_experiment(exp_id)

exp = server.get_experiment_status(exp_id)
print("Status:", exp.status.value, "| global model:", exp.global_model.weights)

Registered node A
Registered node B
Registered node C
Created exp-1: 1 rounds, learning rate 1.0
Started exp-1 with 3 nodes
Round 1
  A: got [0.0] -> trained -> [10.0]
  B: got [0.0] -> trained -> [20.0]
  C: got [0.0] -> trained -> [30.0]
  global model = [20.0]
Status: COMPLETED | global model: [20.0]


## admin console

In [24]:
class AdminConsole:
    """The keyboard: turns typed commands into calls on the server."""

    HELP = """Commands:
  register <node_id> <numbers>   e.g. register A 8,10,12
  create <rounds> <lr>           e.g. create 3 0.5
  start <exp_id>                 e.g. start exp-1
  status <exp_id>                e.g. status exp-1
  help
  quit"""

    def __init__(self, server: FLServer):
        self._server = server

    def register_node(self, node_id: str, data: list[float], address: str = "local") -> None:
        info = NodeInfo(node_id, address)
        worker = WorkerNode(info, data)   # Phase 1: the console creates the simulated node
        self._server.register_node(info, worker)

    def create_experiment(self, rounds: int, lr: float) -> None:
        exp_id = self._server.create_experiment(rounds, lr)
        print(f"-> to run it, type: start {exp_id}")

    def start_experiment(self, exp_id: str) -> None:
        self._server.start_experiment(exp_id)

    def get_status(self, exp_id: str) -> None:
        exp = self._server.get_experiment_status(exp_id)
        print(f"{exp.exp_id}: {exp.status.value}, round {exp.current_round}/{exp.total_rounds}, "
              f"global model {[round(w, 2) for w in exp.global_model.weights]}")

    def start_cli(self) -> None:
        print(self.HELP)
        while True:
            parts = input("fl> ").split()
            if not parts:
                continue
            command, args = parts[0], parts[1:]
            try:
                if command == "register":
                    self.register_node(args[0], [float(x) for x in args[1].split(",")])
                elif command == "create":
                    self.create_experiment(int(args[0]), float(args[1]))
                elif command == "start":
                    self.start_experiment(args[0])
                elif command == "status":
                    self.get_status(args[0])
                elif command == "help":
                    print(self.HELP)
                elif command == "quit":
                    print("Bye!")
                    break
                else:
                    print(f"Unknown command '{command}'. Type help to see the commands.")
            except IndexError:
                print("Missing information. Type help to see an example.")
            except ValueError as e:
                print(f"Error: {e}")

In [25]:
console = AdminConsole(FLServer())

console.register_node("A", [8, 10, 12])
console.register_node("B", [18, 20, 22])
console.register_node("C", [28, 30, 32])

console.create_experiment(rounds=1, lr=1.0)
console.start_experiment("exp-1")
console.get_status("exp-1")

Registered node A
Registered node B
Registered node C
Created exp-1: 1 rounds, learning rate 1.0
-> to run it, type: start exp-1
Started exp-1 with 3 nodes
Round 1
  A: got [0.0] -> trained -> [10.0]
  B: got [0.0] -> trained -> [20.0]
  C: got [0.0] -> trained -> [30.0]
  global model = [20.0]
exp-1: COMPLETED, round 1/1, global model [20.0]


In [27]:
AdminConsole(FLServer()).start_cli()

Commands:
  register <node_id> <numbers>   e.g. register A 8,10,12
  create <rounds> <lr>           e.g. create 3 0.5
  start <exp_id>                 e.g. start exp-1
  status <exp_id>                e.g. status exp-1
  help
  quit


fl>  register A 8,10,12


Registered node A


fl>  register B 18,20,22


Registered node B


fl>  register C 28,30,32


... (7 outputs left out)

fl>  status exp-1


exp-1: COMPLETED, round 3/3, global model [17.5]


fl>  quit


Bye!
